In [1]:
from pathlib import Path

candidates = list(
    Path("/kaggle/input").glob("**/md6_results/md6_best_model")
)

print(candidates)

[PosixPath('/kaggle/input/notebooks/malindus/md6-distilroberta-soft-weighted-ce-alpha075/md6_results/md6_best_model')]


In [2]:
assert len(candidates) == 1, candidates

MODEL_DIR = str(candidates[0])

print("Using MD6 model from:")
print(MODEL_DIR)

Using MD6 model from:
/kaggle/input/notebooks/malindus/md6-distilroberta-soft-weighted-ce-alpha075/md6_results/md6_best_model


In [3]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_DIR
)

print("MD6 loaded successfully")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

MD6 loaded successfully


In [4]:
import json
import numpy as np
import torch

from pathlib import Path
from datasets import Dataset
from transformers import (
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)
from sklearn.metrics import accuracy_score, f1_score, classification_report

# ============================================================
# MD6 VALIDATION VERIFICATION
# ============================================================

VAL_PATH = Path(
    "/kaggle/input/datasets/malindus/mathdial-move-selection/validation.jsonl"
)

LABELS = ["generic", "probing", "focus", "telling"]
LABEL2ID = {label: i for i, label in enumerate(LABELS)}
ID2LABEL = {i: label for label, i in LABEL2ID.items()}

MAX_LENGTH = 512

# Same tokenizer behaviour used by MD6
tokenizer.truncation_side = "left"

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

val_rows = load_jsonl(VAL_PATH)

print("Validation examples:", len(val_rows))
assert len(val_rows) == 1850


# ------------------------------------------------------------
# EXACT MD6 INPUT FORMAT
# ------------------------------------------------------------

def format_history(history):
    if not history:
        return "[No previous conversation]"

    return "\n".join(
        f"{turn['user']}: {turn['text']}"
        for turn in history
    )

def build_problem_text(example):
    return "Problem:\n" + str(example["problem"])

def build_conversation_text(example):
    return (
        "Conversation:\n"
        + format_history(example["history"])
        + "\n\nNext teacher pedagogical move:"
    )


prepared = []

for ex in val_rows:
    prepared.append({
        "problem_text": build_problem_text(ex),
        "conversation_text": build_conversation_text(ex),
        "labels": LABEL2ID[ex["target_move"]],
    })

val_ds = Dataset.from_list(prepared)


def tokenize_batch(batch):
    return tokenizer(
        batch["problem_text"],
        batch["conversation_text"],
        truncation="only_second",
        max_length=MAX_LENGTH,
    )

val_tok = val_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=[
        "problem_text",
        "conversation_text",
    ],
)

assert max(
    len(x) for x in val_tok["input_ids"]
) <= MAX_LENGTH

print("Tokenization: PASSED")


# ------------------------------------------------------------
# PREDICTION
# ------------------------------------------------------------

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

eval_args = TrainingArguments(
    output_dir="/kaggle/working/md6_final_validation_check",
    per_device_eval_batch_size=16,
    fp16=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=eval_args,
    processing_class=tokenizer,
    data_collator=data_collator,
)

pred_output = trainer.predict(val_tok)

logits = pred_output.predictions
true_ids = pred_output.label_ids

pred_ids = np.argmax(logits, axis=1)

accuracy = accuracy_score(
    true_ids,
    pred_ids,
)

macro_f1 = f1_score(
    true_ids,
    pred_ids,
    labels=[0, 1, 2, 3],
    average="macro",
    zero_division=0,
)

per_class_f1 = f1_score(
    true_ids,
    pred_ids,
    labels=[0, 1, 2, 3],
    average=None,
    zero_division=0,
)

print("\n" + "=" * 70)
print("MD6 VALIDATION VERIFICATION")
print("=" * 70)

print("Accuracy :", accuracy)
print("Macro-F1:", macro_f1)

print("\nPer-class F1:")
for label, score in zip(LABELS, per_class_f1):
    print(f"{label:10s}: {score:.10f}")

print("\nClassification report:")
print(
    classification_report(
        true_ids,
        pred_ids,
        target_names=LABELS,
        digits=6,
        zero_division=0,
    )
)

# ------------------------------------------------------------
# VERIFY AGAINST SAVED MD6 EPOCH-2 RESULT
# ------------------------------------------------------------

EXPECTED_ACC = 0.5016216216216216
EXPECTED_F1 = 0.487684572655523

acc_diff = abs(accuracy - EXPECTED_ACC)
f1_diff = abs(macro_f1 - EXPECTED_F1)

print("\nExpected MD6:")
print("Accuracy :", EXPECTED_ACC)
print("Macro-F1:", EXPECTED_F1)

print("\nDifference:")
print("Accuracy difference :", acc_diff)
print("Macro-F1 difference:", f1_diff)

assert acc_diff < 1e-6, (
    "Accuracy does not reproduce the saved MD6 result."
)

assert f1_diff < 1e-6, (
    "Macro-F1 does not reproduce the saved MD6 result."
)

print("\n✅ MD6 VALIDATION REPRODUCTION PASSED")
print("The saved model is verified.")
print("Do NOT load the test set until this message appears.")

Validation examples: 1850


Map:   0%|          | 0/1850 [00:00<?, ? examples/s]

Tokenization: PASSED


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



MD6 VALIDATION VERIFICATION
Accuracy : 0.5016216216216216
Macro-F1: 0.487684572655523

Per-class F1:
generic   : 0.5875000000
probing   : 0.4178712221
focus     : 0.5710080942
telling   : 0.3743589744

Classification report:
              precision    recall  f1-score   support

     generic   0.656425  0.531674  0.587500       442
     probing   0.455587  0.385922  0.417871       412
       focus   0.569750  0.572271  0.571008       678
     telling   0.316017  0.459119  0.374359       318

    accuracy                       0.501622      1850
   macro avg   0.499445  0.487247  0.487685      1850
weighted avg   0.521419  0.501622  0.507042      1850


Expected MD6:
Accuracy : 0.5016216216216216
Macro-F1: 0.487684572655523

Difference:
Accuracy difference : 0.0
Macro-F1 difference: 0.0

✅ MD6 VALIDATION REPRODUCTION PASSED
The saved model is verified.
Do NOT load the test set until this message appears.


In [6]:
import json
import numpy as np

from pathlib import Path
from datasets import Dataset

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

# ============================================================
# FINAL ONE-TIME MD6 HELD-OUT TEST EVALUATION
# ============================================================

TEST_PATH = Path(
    "/kaggle/input/datasets/malindus/mathdial-move-selection/test.jsonl"
)

RESULT_DIR = Path(
    "/kaggle/working/md6_final_test_results"
)
RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("=" * 72)
print("FINAL MD6 HELD-OUT TEST EVALUATION")
print("=" * 72)

print(
    "IMPORTANT: This is the final one-time evaluation."
)

# ------------------------------------------------------------
# LOAD TEST SET
# ------------------------------------------------------------

test_rows = load_jsonl(TEST_PATH)

print("Test examples:", len(test_rows))

assert len(test_rows) == 1852, (
    f"Expected 1852 test examples, got {len(test_rows)}"
)

required = {
    "example_id",
    "qid",
    "problem",
    "history",
    "target_move",
}

assert required.issubset(
    test_rows[0].keys()
), "Unexpected test schema"

bad_labels = [
    ex["target_move"]
    for ex in test_rows
    if ex["target_move"] not in LABEL2ID
]

assert not bad_labels, (
    "Unexpected test labels found."
)

print("Test schema: PASSED")


# ============================================================
# PREPARE EXACT MD6 INPUT
# ============================================================

test_prepared = []

for ex in test_rows:
    test_prepared.append({
        "problem_text": build_problem_text(ex),
        "conversation_text": build_conversation_text(ex),
        "labels": LABEL2ID[
            ex["target_move"]
        ],
    })

test_ds = Dataset.from_list(
    test_prepared
)


# ============================================================
# TOKENIZE EXACTLY AS MD6
# ============================================================

test_tok = test_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=[
        "problem_text",
        "conversation_text",
    ],
)

max_test_length = max(
    len(x)
    for x in test_tok["input_ids"]
)

assert max_test_length <= MAX_LENGTH

print(
    "Maximum tokenized test length:",
    max_test_length,
)

print("Test tokenization: PASSED")


# ============================================================
# FINAL PREDICTION
# ============================================================

print("\nRunning final MD6 test inference...")

test_output = trainer.predict(
    test_tok
)

test_logits = (
    test_output.predictions
)

test_true_ids = (
    test_output.label_ids
)

test_pred_ids = np.argmax(
    test_logits,
    axis=1,
)


# ============================================================
# FINAL METRICS
# ============================================================

test_accuracy = accuracy_score(
    test_true_ids,
    test_pred_ids,
)

test_macro_f1 = f1_score(
    test_true_ids,
    test_pred_ids,
    labels=[0, 1, 2, 3],
    average="macro",
    zero_division=0,
)

test_weighted_f1 = f1_score(
    test_true_ids,
    test_pred_ids,
    labels=[0, 1, 2, 3],
    average="weighted",
    zero_division=0,
)

test_per_class_f1 = f1_score(
    test_true_ids,
    test_pred_ids,
    labels=[0, 1, 2, 3],
    average=None,
    zero_division=0,
)

test_cm = confusion_matrix(
    test_true_ids,
    test_pred_ids,
    labels=[0, 1, 2, 3],
)

test_report = classification_report(
    test_true_ids,
    test_pred_ids,
    labels=[0, 1, 2, 3],
    target_names=LABELS,
    digits=6,
    zero_division=0,
    output_dict=True,
)


# ============================================================
# DISPLAY FINAL RESULTS
# ============================================================

print("\n" + "=" * 72)
print("FINAL HELD-OUT TEST RESULTS — MD6")
print("=" * 72)

print(
    "Accuracy :",
    test_accuracy,
)

print(
    "Macro-F1:",
    test_macro_f1,
)

print(
    "Weighted-F1:",
    test_weighted_f1,
)

print("\nPer-class F1:")

for label, score in zip(
    LABELS,
    test_per_class_f1,
):
    print(
        f"{label:10s}: "
        f"{score:.10f}"
    )

print("\nClassification report:")

print(
    classification_report(
        test_true_ids,
        test_pred_ids,
        labels=[0, 1, 2, 3],
        target_names=LABELS,
        digits=6,
        zero_division=0,
    )
)

print("\nConfusion matrix")
print(
    "Rows = true labels"
)
print(
    "Columns = predicted labels"
)
print(
    "Label order:",
    LABELS,
)
print(test_cm)


# ============================================================
# SOFTMAX PROBABILITIES
# ============================================================

def softmax_np(logits):
    shifted = (
        logits
        - logits.max(
            axis=1,
            keepdims=True,
        )
    )

    exp_values = np.exp(
        shifted
    )

    return (
        exp_values
        / exp_values.sum(
            axis=1,
            keepdims=True,
        )
    )

test_probs = softmax_np(
    test_logits
)


# ============================================================
# SAVE TEST PREDICTIONS
# ============================================================

prediction_path = (
    RESULT_DIR
    / "md6_final_test_predictions.jsonl"
)

with prediction_path.open(
    "w",
    encoding="utf-8",
) as f:

    for i, (
        true_id,
        pred_id,
        probability_row,
    ) in enumerate(
        zip(
            test_true_ids,
            test_pred_ids,
            test_probs,
        )
    ):

        record = {
            "example_id": (
                test_rows[i]["example_id"]
            ),

            "qid": (
                test_rows[i]["qid"]
            ),

            "true_move": (
                ID2LABEL[
                    int(true_id)
                ]
            ),

            "predicted_move": (
                ID2LABEL[
                    int(pred_id)
                ]
            ),

            "probabilities": {
                label: float(
                    probability_row[j]
                )
                for j, label
                in enumerate(LABELS)
            },
        }

        f.write(
            json.dumps(record)
            + "\n"
        )


# ============================================================
# SAVE FINAL SUMMARY
# ============================================================

summary = {
    "experiment": (
        "MD6_Final_Held_Out_Test"
    ),

    "model": (
        "distilbert/distilroberta-base"
    ),

    "checkpoint": (
        "verified MD6 epoch 2"
    ),

    "model_selection_complete_before_test": True,

    "test_examples": len(
        test_rows
    ),

    "validation_verification": {
        "accuracy": (
            0.5016216216216216
        ),
        "macro_f1": (
            0.487684572655523
        ),
        "exact_reproduction": True,
    },

    "test_results": {
        "accuracy": float(
            test_accuracy
        ),

        "macro_f1": float(
            test_macro_f1
        ),

        "weighted_f1": float(
            test_weighted_f1
        ),

        "per_class_f1": {
            label: float(score)
            for label, score
            in zip(
                LABELS,
                test_per_class_f1,
            )
        },

        "confusion_matrix": (
            test_cm.tolist()
        ),

        "classification_report": (
            test_report
        ),
    },
}

summary_path = (
    RESULT_DIR
    / "md6_final_test_summary.json"
)

with summary_path.open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        summary,
        f,
        indent=2,
    )


# ============================================================
# FINAL MESSAGE
# ============================================================

print("\n" + "=" * 72)
print("FINAL TEST EVALUATION COMPLETE")
print("=" * 72)

print(
    "Predictions saved to:"
)
print(prediction_path)

print(
    "\nSummary saved to:"
)
print(summary_path)

print(
    "\nIMPORTANT:"
)
print(
    "The held-out test set has now been consumed."
)
print(
    "Do not tune or select further supervised models "
    "using these test results."
)

FINAL MD6 HELD-OUT TEST EVALUATION
IMPORTANT: This is the final one-time evaluation.
Test examples: 1852
Test schema: PASSED


Map:   0%|          | 0/1852 [00:00<?, ? examples/s]

Maximum tokenized test length: 512
Test tokenization: PASSED

Running final MD6 test inference...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



FINAL HELD-OUT TEST RESULTS — MD6
Accuracy : 0.519438444924406
Macro-F1: 0.50938899285248
Weighted-F1: 0.5222339194094733

Per-class F1:
generic   : 0.6218274112
probing   : 0.3984476067
focus     : 0.5685131195
telling   : 0.4487678340

Classification report:
              precision    recall  f1-score   support

     generic   0.694051  0.563218  0.621827       435
     probing   0.445087  0.360656  0.398448       427
       focus   0.569343  0.567686  0.568513       687
     telling   0.369658  0.570957  0.448768       303

    accuracy                       0.519438      1852
   macro avg   0.519535  0.515629  0.509389      1852
weighted avg   0.537316  0.519438  0.522234      1852


Confusion matrix
Rows = true labels
Columns = predicted labels
Label order: ['generic', 'probing', 'focus', 'telling']
[[245  30  78  82]
 [ 52 154 131  90]
 [ 47 127 390 123]
 [  9  35  86 173]]

FINAL TEST EVALUATION COMPLETE
Predictions saved to:
/kaggle/working/md6_final_test_results/md6_final_tes